### 导入库 打印目录

In [ ]:
import os
import warnings

import numpy as np 
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import  accuracy_score,confusion_matrix,classification_report
from sklearn.metrics import accuracy_score,classification_report 

import torch
import torch.nn as nn
import torch.optim as optim 
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader

from tqdm import tqdm
import time

plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"]=False 

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### 定义全局参数

In [ ]:
class Config:
    def __init__(self):
        self.seed = 666  # 随机种子
        
        # 训练参数
        self.epochs = 666  # 轮数
        self.batch_size = 50  # 批次大小（
        self.val_ratio = 0.2  # 分割比例
        self.val_batch_size = 30  # 验证集批次大小
        self.learning_rate = 0.001  # 学习率
        self.weight_decay = 0.00001  # 权重衰减（
        self.lr_scheduler = True  # 是否启用学习率调度器
        self.scheduler_patience = 5  # 调度patience
        self.early_stopping = True
        self.early_stop_patience = 50 # 早停patience

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.num_workers = 16 

        # 图像参数
        self.image_cols = None  # 调用update_from_dataset函数检测确定
        self.label_col = "label"  # 标签列名
        self.image_height = 28  # 图像高度
        self.image_width = 28  # 图像宽度
        self.num_channels = 1  # 通道数（1为灰度图，3为RGB）
        self.input_size = None  # 输入维度-调用下面函数计算
        self._calculate_input_size()  # 计算输入维度
        self.num_classes = 10  #输出分类

        #预处理
        self.normalize = True
        self.shuffle = True
        self.stratify  =True

        # 输入
        self.data_path = "/kaggle/input/"  # 数据根路径
        self.train_csv_path = os.path.join(self.data_path, "digit-recognizer/train.csv")  # 训练数据CSV路径
        self.image_dir = os.path.join(self.data_path, "images/")  # 图像文件夹路径
        
        # 输出
        self.output_dir = "/kaggle/working/outputs/"  # 输出根目录
        self.model_save_path = os.path.join(self.output_dir, "best_model/")  # 模型保存路径
        self.submission_path = os.path.join(self.output_dir, "submission/")  # 提交文件路径
        

        os.makedirs(self.output_dir, exist_ok=True)

        

    def _calculate_input_size(self):
        self.input_size = self.image_height * self.image_width * self.num_channels  
        

    def update_from_dataset(self, df):
        # 检查标签列是否存在
        if self.label_col in df.columns:
            print(f"标签列: {self.label_col}")
        else:
            print("标签列不存在")

        # 自动检测图像特征列
        if self.image_cols is None:
            exclude_cols = [self.label_col, 'id', 'index']  # 排除非特征列
            self.image_cols = [col for col in df.columns if col not in exclude_cols]
            print(f"自动检测到 {len(self.image_cols)} 个图像特征列")

        # 确定类别数量
        if self.label_col in df.columns:
            self.num_classes = len(df[self.label_col].unique())
            print(f"类别数量: {self.num_classes}")
        else:
            print("标签列不存在，无法确定类别数量")

    def print_info(self):
        print("=" * 50)
        print(f"设备: {self.device}")
        print(f"随机种子: {self.seed}")
        print(f"训练轮数: {self.epochs}")
        print(f"批次大小: 训练={self.batch_size}, 验证={self.val_batch_size}")
        print(f"学习率: {self.learning_rate}")
        print(f"图像尺寸: {self.image_height}x{self.image_width} (通道数: {self.num_channels})")
        print(f"输入维度: {self.input_size}")
        print(f"类别数量: {self.num_classes}")
        print(f"训练数据路径: {self.train_csv_path}")
        print(f"输出目录: {self.output_dir}")
        print("=" * 50)
    

In [ ]:
CONFIG = Config()
print(f"使用设备：{CONFIG.device}")

In [ ]:
CONFIG.print_info()

### 定义随机种子

In [ ]:
def set_random_seed(seed=CONFIG.seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        #为当前GPU设定种子
        torch.cuda.manual_seed_all(seed)
        #如果存在多个GPU 设定为同一个种子
        torch.backends.cudnn.deterministic = True
        #设置cuDNN使用确定性算法，这样每次运行的结果都会一致。但注意，这可能会降低性能。
        torch.backends.cudnn.benchmark = False
        #关闭cuDNN的benchmark模式。benchmark模式会根据硬件和输入大小选择最优的算法，
        #但这样会导致每次运行可能选择不同的算法，从而造成结果不一致。关闭它可以确保确定性。

In [ ]:
set_random_seed()
print(f"随即种子已设定：{CONFIG.seed}")

In [ ]:
train_look_df = pd.read_csv(CONFIG.train_csv_path)
train_look_df.head(10)

In [ ]:
# train_data,val_data = load_and_split_data(CONFIG)

### 加载和预处理数据

In [ ]:
def load_and_preprocess_data(config):
    
    if not os.path.exists(config.train_csv_path):
        raise FileNotFoundError(f"训练文件数据不存在：{config.train_csv_path}")
    
    train_path = config.train_csv_path
    train_df = pd.read_csv(train_path)
    print(f"原始训练数据形状：{train_df.shape}")
    
    config.update_from_dataset(train_df)

    if config.label_col in train_df.columns:
        x = train_df.drop(config.label_col,axis = 1).values
        y = train_df[config.label_col].values
    else:
        print("傻逼去看看你的标签，没找到分什么分")
        return 

    if config.normalize:
        x = x.astype(np.float32)/255.0
        print("归一化[0,1]了")

    if y is not None:
        stratify = y if config.stratify else None

        x_train,x_val,y_train,y_val = train_test_split(
            x,y,test_size = config.val_ratio,
            shuffle = config.shuffle,stratify = stratify
        )
    print(f"训练集：{len(x_train)}条，验证集：{len(x_val)}条")
    return (x_train,y_train),(x_val,y_val)

    

In [ ]:
train_data,val_data = load_and_preprocess_data(CONFIG)
x_train,y_train = train_data
x_val,y_val  = val_data

### 封装成自定义数据集类

In [ ]:
class CustomDataset(Dataset):  
    def __init__(self, features, labels=None, transform=None): 
        self.features = features  
        self.labels = labels      
        self.transform = transform 

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        #获取单个样本（特征+标签，或仅特征）
        # 获取特征并转换为张量
        feature = self.features[idx]
        feature_tensor = torch.tensor(feature, dtype=torch.float32)
        
        if self.transform:
            feature_tensor = self.transform(feature_tensor)

        if self.labels is not None:  
            label = self.labels[idx]
            label_tensor = torch.tensor(label, dtype=torch.long)
            return feature_tensor, label_tensor
        else:
            print("检查标签")
            return feature_tensor
    

### 创建训练和验证数据加载器

In [ ]:
def get_dataloaders(config,train_data,val_data):
    x_train,y_train = train_data
    x_val,y_val = val_data

    #创建数据集
    train_dataset = CustomDataset(x_train,y_train)
    val_dataset = CustomDataset(x_val,y_val)

    #创建数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle = config.shuffle,
        num_workers = 0, #默认值 指定用于加载数据的子进程数量
        pin_memory = True 
        #开启 “固定内存” 模式，当使用 GPU 时，会将加载到 CPU 的数据直接复制到 CUDA 固定内存中，
        #减少 CPU 到 GPU 的数据传输时间，提高效率。
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle = False,
        num_workers = 0,
        pin_memory = True
    )
    print("数据加载器创建完成：")
    print(f"  训练集: {len(train_dataset)} 样本, {len(train_loader)} 批次")
    print(f"  验证集: {len(val_dataset)} 样本, {len(val_loader)} 批次")

    return train_loader,val_loader

In [ ]:
train_loader,val_loader = get_dataloaders(CONFIG,train_data,val_data)

### 训练模型

In [ ]:
class MLP(nn.Module):
    def __init__(self,dropout_rate = 0.1):
        super(MLP,self).__init__()
        self.l1 = nn.Sequential(
            nn.Linear(28*28, 2048, bias=True), 
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True), 
            nn.Dropout(dropout_rate)    #使用dropout时候epoch要大一点
        )

        self.l2 = nn.Sequential(
            nn.Linear(2048, 512, bias=True),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate)   
        )
        
        self.l3 = nn.Sequential(
            nn.Linear(512, 10, bias=True),
            nn.BatchNorm1d(10),
            nn.ReLU(inplace=True), 
            nn.Dropout(dropout_rate)   
        )
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        
        return x 

In [ ]:
mlp_model = MLP()
mlp_model = mlp_model.to(CONFIG.device)  # 同样要转换device

### 训练函数

In [ ]:
def train_epoch(model,train_loader,criterion,optimizer,device):
    model.train() #切换至训练模式，与eval()相对应
    running_loss = 0.0 #累计损失
    correct_predictions = 0# 累计正确预测数
    total_samples = 0 # 累计总样本

    for features,labels in tqdm(train_loader,desc='训练ing...'):
        #遍历数据加载器的每一个批次
        features = features.to(device)
        labels = labels.to(device)

        #梯度清零
        optimizer.zero_grad()

        #前向传播
        outputs = model(features)
        loss = criterion(outputs,labels)
        #反向传播
        loss.backward() #自动计算所有可训练参数的梯度
        optimizer.step() #根据梯度更新参数
        
        running_loss+=loss.item()*features.size(0)
        _,preds = torch.max(outputs,1)
        correct_predictions+=torch.sum(preds==labels.data)
        total_samples+=features.size(0)
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions.double() / total_samples

    return epoch_loss,epoch_acc.item()

### 验证函数

In [ ]:
def validate_epoch(model, val_loader, criterion, device):
    model.eval()  # 切换到评估模式
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0 

    with torch.no_grad():  
        for features, labels in tqdm(val_loader, desc='验证ing...'):
            features = features.to(device)
            labels = labels.to(device)

            # 前向传播
            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)  
            _, preds = torch.max(outputs, 1)  # 预测类别
            correct_predictions += torch.sum(preds == labels.data)  
            total_samples += features.size(0) 
        
        # 循环结束后计算整个epoch的平均损失和准确率
        epoch_loss = running_loss / total_samples
        epoch_acc = correct_predictions.double() / total_samples

        return epoch_loss, epoch_acc.item()

### 完整训练函数

In [ ]:
def train_model(model, train_loader, val_loader, config):
    criterion = nn.CrossEntropyLoss() #交叉熵
    # 优化器（Adam带L2正则化）
    optimizer = optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    scheduler = None
    if config.lr_scheduler:  
        scheduler = optim.lr_scheduler.ReduceLROnPlateau( 
            optimizer,
            mode='max',  # 监控验证准确率最大化 要是 loss 则改min
            factor=0.5,  # 学习率衰减为原来的1/2
            patience=config.scheduler_patience,  # 连续patience轮无提升则衰减
            verbose=True  
        )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'learning_rates': [] 
    }

    # 早停
    best_val_acc = 0.0
    best_model_state = None
    epochs_no_improve = 0
    
    print(f"开始训练，共{config.epochs}轮")
    start_time = time.time()

    for epoch in range(config.epochs):
        print(f"\n {epoch+1}/{config.epochs}") 
        print("-" * 30)

        current_lr = optimizer.param_groups[0]['lr']
        history['learning_rates'].append(current_lr)
        print(f"当前学习率：{current_lr:.6f}")

        # 训练一个epoch
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, config.device)
        # 验证一个epoch
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, config.device)
    
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        print(f"训练集  损失: {train_loss:.4f}, 准确率: {train_acc:.4f}")
        print(f"验证集  损失: {val_loss:.4f}, 准确率: {val_acc:.4f}")

        if scheduler:
            scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()  # 深拷

            save_path = os.path.join(config.model_save_path, "3层mlp_best_model.pth")  
            save_dir = os.path.dirname(save_path)  
            os.makedirs(save_dir, exist_ok=True)  # 创建目录 
            
            torch.save(best_model_state, save_path)
            print(f"最佳模型权重已保存至：{save_path}，验证准确率：{best_val_acc:.4f}")
        else:
            epochs_no_improve += 1
            print(f"验证准确率未提升，已持续 {epochs_no_improve} 轮")

        # 早停检查
        if config.early_stopping and epochs_no_improve >= config.early_stop_patience:
            print(f"第 {epoch+1} 轮无提升，已早停") 
            break

    return best_val_acc, history

In [ ]:
best_val_acc,history = train_model(mlp_model, train_loader, val_loader, CONFIG)